In [ ]:
import torch

head = 4
head_dim = 16
dim = head * head_dim
seq_len = 32
num_key_value_heads = 2

x = torch.randint(0, 10, (1, dim))  # (1, dim)
wq = torch.randint(0, 10, (dim, dim))  # (dim, dim)
wk = torch.randint(0, 10, (dim, num_key_value_heads * head_dim))  # (dim, dim)
xt = torch.randint(0, 10, (seq_len, dim))  # (seq_l en, dim)

q = torch.matmul(x, wq)  # (1, dim)
k = torch.matmul(xt, wk)  # (seq_len, dim)

q_reshape = q.reshape(1, head, head_dim).transpose(0, 1)  # (head, 1, head_dim)
k_reshape = k.reshape(seq_len, head, head_dim).transpose(0, 1).transpose(-1, -2)  # (head, head_dim, seq_len)

expect = torch.matmul(q_reshape, k_reshape)  # (head, 1, seq_len)
print(expect.shape)

wk_reshape = wk.transpose(0, 1).reshape(head, head_dim, dim)  # (head, head_dim, dim)

qwk = torch.matmul(q_reshape, wk_reshape)  # (head, 1, dim)

#qwk = qwk.sum(0)
result = torch.matmul(qwk, xt.transpose(0, 1))  # (head, 1, seq_len)

print(result.shape)

if torch.allclose(expect, result, rtol=1e-2, atol=1e-2):
    print(expect)
    print(result)
    print(1)

torch.Size([4, 1, 32])
torch.Size([4, 1, 32])
tensor([[[29577987, 29237280, 25264886, 23423141, 24809977, 26438696, 24018442,
          24347662, 24473287, 25480767, 22511857, 27657782, 28293008, 25161031,
          27308147, 23745579, 28095684, 24584005, 22873646, 27775698, 26804309,
          28466540, 24696968, 25013893, 22465637, 27060708, 25797800, 24151147,
          23168496, 24143692, 23270740, 24053143]],

        [[27844144, 28145275, 24356298, 22576747, 23974756, 25587023, 23226426,
          24274200, 23435449, 25148449, 21792957, 27031671, 27577718, 23588648,
          26195013, 23138474, 25758070, 23767613, 21819154, 26927881, 26060708,
          28174031, 23764448, 23995676, 21203513, 26455392, 24899307, 22946523,
          23001098, 23654460, 22995045, 23279647]],

        [[28751588, 28724929, 24715228, 23275027, 24926750, 25683365, 23937679,
          23829009, 23811839, 25890935, 22226662, 28025999, 27674671, 24619416,
          27319707, 23239969, 27540633, 25319792

In [6]:
import torch
import torch.nn.functional as F

head = 4
head_dim = 16
dim = head * head_dim
seq_len = 32
num_key_value_heads = 2

# GQA 要求 query 头的数量必须是 key/value 头的整数倍
assert head % num_key_value_heads == 0
num_key_value_groups = head // num_key_value_heads

x = torch.randint(0, 10, (1, dim), dtype=torch.float32)  # (1, dim)
wq = torch.randint(0, 10, (dim, dim), dtype=torch.float32)  # (dim, dim)
wk = torch.randint(0, 10, (dim, num_key_value_heads * head_dim), dtype=torch.float32)  # (dim, kv_dim)
wv = torch.randn(dim, num_key_value_heads * head_dim, dtype=torch.float32)
xt = torch.randint(0, 10, (seq_len, dim), dtype=torch.float32)  # (seq_len, dim)

# --- 正确的 GQA 计算方式 (expect) ---
q = torch.matmul(x, wq)  # (1, dim)
k = torch.matmul(xt, wk)  # (seq_len, kv_dim)
v = torch.matmul(xt, wv)  # (seq_len, kv_dim)

# Reshape Q
q_reshape = q.view(1, head, head_dim).transpose(0, 1)  # (head, 1, head_dim)

# Reshape K
k_reshape = k.view(seq_len, num_key_value_heads, head_dim) # (seq_len, num_kv_heads, head_dim)

v_reshape = v.view(seq_len, num_key_value_heads, head_dim) # (seq_len, num_kv_heads, head_dim)
v_repeated = v_reshape.repeat_interleave(num_key_value_groups, dim=1) # (seq_len, head, head_dim)

# 重复K的头以匹配Q的头
k_repeated = k_reshape.repeat_interleave(num_key_value_groups, dim=1) # (seq_len, head, head_dim)

# 转置K以进行矩阵乘法
k_transposed = k_repeated.transpose(0, 1).transpose(-1, -2)  # (head, head_dim, seq_len)

expect = torch.matmul(q_reshape, k_transposed)  # (head, 1, seq_len)
print("Expect shape:", expect.shape)

# --- 您的分解计算方式 (result) ---
wk_reshape = wk.transpose(0, 1).reshape(num_key_value_heads, head_dim, dim)  # (num_kv_heads, head_dim, dim)
wk_repeated = wk_reshape.repeat_interleave(num_key_value_groups, dim=0) # (head, head_dim, dim)

qwk = torch.matmul(q_reshape, wk_repeated)  # (head, 1, dim)

result = torch.matmul(qwk, xt.transpose(0, 1))  # (head, 1, seq_len)

print("Result shape:", result.shape)

if torch.allclose(expect, result, rtol=1e-5, atol=1e-5):
    print("\nSuccess: The results are close enough.")
    # print("Expect:\n", expect)
    # print("Result:\n", result)
else:
    print("\nFailure: The results are different.")
    print("Max difference:", (expect - result).abs().max())
    
p = F.softmax(expect, dim=-1)
print(p.shape)
print(v_repeated.shape)

# p shape(head, 1, seq_len)
# x shape(seq_len, head, head_dim)

expect_output = torch.matmul(p, v_repeated.transpose(0, 1))  # (head, 1, head_dim)

print("Expect Output shape:", expect_output.shape)

# --- TCD (pX)Wv 计算方式 (result_output) ---
# 这是你要求的 (p @ X) @ Wv 计算顺序
# p shape: (head, 1, seq_len)
# xt (即 X) shape: (seq_len, dim)

# 1. (p @ X)
# (head, 1, seq_len) @ (seq_len, dim) -> (head, 1, dim)
pX = torch.matmul(p, xt)

# 2. 准备 Wv
# wv shape: (dim, kv_dim) = (dim, num_key_value_heads * head_dim)
# 我们需要将其 reshape 并 repeat，以匹配 Q 的头数 (head)

# wv_reshape: (dim, num_kv_heads, head_dim)
wv_reshape = wv.view(dim, num_key_value_heads, head_dim)

# wv_repeated: (dim, head, head_dim)
wv_repeated = wv_reshape.repeat_interleave(num_key_value_groups, dim=1)

# 3. (pX) @ Wv_repeated
#    我们需要 (head, 1, dim) @ (dim, head, head_dim)
#    这需要将 head 维度作为 "batch" 维度

# pX_b (pX_batched): (head, 1, dim)
# (pX 已经是正确的 bmm 形状了)
pX_b = pX

# wv_repeated_b (wv_repeated_batched): (head, dim, head_dim)
wv_repeated_b = wv_repeated.permute(1, 0, 2)

# (head, 1, dim) @ (head, dim, head_dim) -> (head, 1, head_dim)
result_output = torch.bmm(pX_b, wv_repeated_b)

print("Result Output shape:", result_output.shape)

# --- 最终验证 ---
if torch.allclose(expect_output, result_output, rtol=1e-5, atol=1e-5):
    print("\nSuccess: Output results are close enough.")
else:
    print("\nFailure: Output results are different.")
    print("Max difference:", (expect_output - result_output).abs().max())

Expect shape: torch.Size([4, 1, 32])
Result shape: torch.Size([4, 1, 32])

Success: The results are close enough.
torch.Size([4, 1, 32])
torch.Size([32, 4, 16])
Expect Output shape: torch.Size([4, 1, 16])
Result Output shape: torch.Size([4, 1, 16])

Success: Output results are close enough.
